# 03 — CHM Derivation: DSM, DTM, and Canopy Height Model

## What this notebook does

This notebook rasterizes the normalized point cloud from Notebook 02 into
three raster products:

1. **DSM** — Digital Surface Model: max return height per pixel (top of canopy)
2. **DTM** — Digital Terrain Model: ground-classified returns only (bare earth)
3. **CHM** — Canopy Height Model: DSM minus DTM (vegetation height above ground)

The CHM GeoTIFF is the primary output of this notebook and the input to
the corridor analysis in Notebook 04.

---

## Inputs expected

- `data/processed/<tile_name>_normalized.laz` — normalized point cloud
  produced by Notebook 02, with ground classification (class 2) and
  HeightAboveGround dimension assigned.

## Outputs produced

- `data/processed/dsm.tif` — DSM GeoTIFF at target resolution
- `data/processed/dtm.tif` — DTM GeoTIFF at target resolution
- `data/processed/chm.tif` — Full-extent CHM GeoTIFF (DSM − DTM)

## src/ functions called

- `src.chm.rasterize_to_dsm(normalized_laz, resolution, output_path)`
- `src.chm.rasterize_to_dtm(normalized_laz, resolution, output_path)`
- `src.chm.compute_chm(dsm_path, dtm_path, output_path)`

---

## Key decisions to make before writing code

- **Target resolution:** 1m is the intended target, but this must be
  validated against the actual point density of the tile. Use the
  density estimate from `inspect_point_cloud()` in Notebook 01.
  At QL2 density (~2 pts/m²), 1m resolution should be achievable;
  at lower densities, 2m or 3m may be more appropriate.
- **NoData handling:** Decide how to handle cells with no returns
  (e.g., water, roads). Fill with NoData or interpolate?
- **Negative CHM values:** Subtraction artifacts may produce small
  negative values. Decide whether to clamp to zero or leave as-is
  and document the choice.
- **Rasterization library:** Evaluate whether to use `rioxarray`,
  PDAL's `writers.gdal`, or `scipy`/`numpy` directly for the
  rasterization step. PDAL's built-in writer may be most straightforward.
